# Synthetic Complete Experiment

Runs the full 20-seed synthetic CMDL, Plain LSTM, and ablation suite, then saves table artifacts only.

In [1]:
from argparse import Namespace
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

current = Path.cwd().resolve()
repo_root = next((p for p in [current, *current.parents] if (p / "experiments").exists() and (p / "config").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from {current}")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluation.synthetic_comparison import build_identification_table, build_recovery_table, build_significance_tables, build_synthetic_comparison
from experiments import run_ablation, run_ganet_baseline, run_lstm_baseline, run_synthetic, run_tft_baseline
from experiments.run_complete_20seed_suite import SCENARIOS, SYNTHETIC_VARIANTS, cleanup, synthetic_common_args

PLAN_NAME = "complete_20seed_20260426"
SEEDS = list(range(20))
FORCE = False
RUN_CMDL = True
RUN_BASELINE = True
RUN_TFT = True
RUN_GANET = True
RUN_ABLATIONS = True

OUTPUT_ROOT = repo_root / "outputs" / "notebook_synthetic" / PLAN_NAME
CMDL_DIR = OUTPUT_ROOT / "cmdl"
BASELINE_DIR = OUTPUT_ROOT / "plain_lstm"
TFT_DIR = OUTPUT_ROOT / "tft"
GANET_DIR = OUTPUT_ROOT / "ganet"
ABLATION_DIR = OUTPUT_ROOT / "ablation"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
for path in [CMDL_DIR, BASELINE_DIR, TFT_DIR, GANET_DIR, ABLATION_DIR, COMPARISON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def summary_exists(run_dir: Path) -> bool:
    return (run_dir / "summary.json").exists()

def run_task(label: str, run_dir: Path, callback) -> None:
    if summary_exists(run_dir) and not FORCE:
        print(f"[skip] {label}: {run_dir}")
        return
    if run_dir.exists() and (FORCE or not summary_exists(run_dir)):
        reason = "force rerun" if FORCE else "incomplete artifact"
        print(f"[clean] {reason}: {run_dir}")
        shutil.rmtree(run_dir)
    print(f"[run] {label}: {run_dir}")
    callback()
    cleanup()
    if not summary_exists(run_dir):
        raise RuntimeError(f"Expected summary.json was not created for {label}: {run_dir}")

settings = pd.Series({"plan_name": PLAN_NAME, "seeds": SEEDS, "scenarios": SCENARIOS, "force": FORCE, "output_root": OUTPUT_ROOT}).to_frame("value")
display(settings)

c:\Users\42155\anaconda3\envs\PTenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
plan_name,complete_20seed_20260426
seeds,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
scenarios,"[linear, nonlinear]"
force,False
output_root,C:\DevSpace\PyDevspace\CMDL\outputs\notebook_s...


In [6]:
if RUN_CMDL:
    for seed in SEEDS:
        args = Namespace(**synthetic_common_args(CMDL_DIR), seed=seed)
        for scenario in SCENARIOS:
            name = f"cmdl_{scenario}_seed{seed}"
            run_task(f"synthetic CMDL {scenario} seed {seed}", CMDL_DIR / name, lambda args=args, name=name, scenario=scenario: run_synthetic.run_experiment(args, name, scenario))

if RUN_BASELINE:
    for seed in SEEDS:
        args = Namespace(**synthetic_common_args(BASELINE_DIR), seed=seed)
        for scenario in SCENARIOS:
            name = f"plain_lstm_{scenario}_seed{seed}"
            run_task(f"synthetic Plain LSTM {scenario} seed {seed}", BASELINE_DIR / name, lambda args=args, name=name, scenario=scenario: run_lstm_baseline.run_experiment(args, name, scenario))

if RUN_ABLATIONS:
    ablation_args = Namespace(**synthetic_common_args(ABLATION_DIR), variant="all", seeds=SEEDS)
    for seed in SEEDS:
        for scenario in SCENARIOS:
            for variant in SYNTHETIC_VARIANTS:
                name = f"{variant}_{scenario}_seed{seed}"
                run_task(f"synthetic ablation {variant} {scenario} seed {seed}", ABLATION_DIR / name, lambda variant=variant, scenario=scenario, seed=seed: run_ablation.run_variant(ablation_args, variant, scenario, seed))

[skip] synthetic CMDL linear seed 0: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\cmdl\cmdl_linear_seed0
[skip] synthetic CMDL nonlinear seed 0: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\cmdl\cmdl_nonlinear_seed0
[skip] synthetic CMDL linear seed 1: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\cmdl\cmdl_linear_seed1
[skip] synthetic CMDL nonlinear seed 1: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\cmdl\cmdl_nonlinear_seed1
[skip] synthetic CMDL linear seed 2: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\cmdl\cmdl_linear_seed2
[skip] synthetic CMDL nonlinear seed 2: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\cmdl\cmdl_nonlinear_seed2
[skip] synthetic CMDL linear seed 3: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\cmdl\cmdl_linear_seed3
[s

In [7]:
if RUN_TFT:
    for seed in SEEDS:
        args = Namespace(**synthetic_common_args(TFT_DIR), seed=seed)
        for scenario in SCENARIOS:
            name = f"tft_{scenario}_seed{seed}"
            run_task(f"synthetic TFT {scenario} seed {seed}", TFT_DIR / name, lambda args=args, name=name, scenario=scenario: run_tft_baseline.run_experiment(args, name, scenario))

if RUN_GANET:
    for seed in SEEDS:
        args = Namespace(**synthetic_common_args(GANET_DIR), seed=seed)
        for scenario in SCENARIOS:
            name = f"ganet_{scenario}_seed{seed}"
            run_task(f"synthetic GA-Net {scenario} seed {seed}", GANET_DIR / name, lambda args=args, name=name, scenario=scenario: run_ganet_baseline.run_experiment(args, name, scenario))

[run] synthetic TFT linear seed 0: C:\DevSpace\PyDevspace\CMDL\outputs\notebook_synthetic\complete_20seed_20260426\tft\tft_linear_seed0
[tft_linear_seed0] epoch=001 train_task=0.2552 val_task=0.2379 val_mae=0.3941 val_r2=-0.1036
[tft_linear_seed0] epoch=010 train_task=0.0935 val_task=0.1001 val_mae=0.2480 val_r2=0.5368
[tft_linear_seed0] epoch=020 train_task=0.0747 val_task=0.0803 val_mae=0.2227 val_r2=0.6280
[tft_linear_seed0] epoch=030 train_task=0.0564 val_task=0.0624 val_mae=0.1973 val_r2=0.7080
[tft_linear_seed0] epoch=040 train_task=0.0476 val_task=0.0528 val_mae=0.1812 val_r2=0.7539
[tft_linear_seed0] epoch=050 train_task=0.0408 val_task=0.0468 val_mae=0.1714 val_r2=0.7827
[tft_linear_seed0] epoch=060 train_task=0.0349 val_task=0.0453 val_mae=0.1683 val_r2=0.7902
[tft_linear_seed0] epoch=070 train_task=0.0321 val_task=0.0439 val_mae=0.1640 val_r2=0.7972
[tft_linear_seed0] epoch=080 train_task=0.0289 val_task=0.0434 val_mae=0.1636 val_r2=0.7995
[tft_linear_seed0] epoch=090 train_

In [2]:
SUMMARY_METRICS = [
    "task_loss",
    "effective_kstar_mae",
    "effective_kstar_spearman_rho",
    "effective_lag_entropy_mean",
    "effective_lag_peak_accuracy",
    "proxy_signal_r2",
    "z_signal_spearman_rho",
]

def attach_seed(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    result = frame.copy()
    seed_from_experiment = result["experiment"].astype(str).str.extract(r"seed(\d+)")[0]
    seed_from_run_dir = result["run_dir"].astype(str).str.extract(r"seed(\d+)")[0]
    result["seed"] = pd.to_numeric(seed_from_experiment.fillna(seed_from_run_dir), errors="coerce").astype("Int64")
    return result

def aggregate_multiseed(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=["display_name", "family", "variant", "scenario", "n_seeds"])
    available = [column for column in SUMMARY_METRICS if column in frame.columns]
    group_columns = ["display_name", "family", "variant", "scenario"]
    working = frame.copy()
    for column in available:
        working[column] = pd.to_numeric(working[column], errors="coerce")
    grouped = working.groupby(group_columns, dropna=False)
    summary = grouped[available].agg(["mean", "std"]).reset_index()
    summary.columns = [column if isinstance(column, str) else "_".join(part for part in column if part) for column in summary.columns.to_flat_index()]
    summary = summary.merge(grouped["seed"].nunique(dropna=True).reset_index(name="n_seeds"), on=group_columns, how="left")
    if "effective_kstar_spearman_rho" in available:
        shares = grouped["effective_kstar_spearman_rho"].apply(lambda values: float((pd.to_numeric(values, errors="coerce").dropna() > 0).mean()) if not pd.to_numeric(values, errors="coerce").dropna().empty else np.nan).reset_index(name="kstar_positive_seed_share")
        summary = summary.merge(shares, on=group_columns, how="left")
    return summary.sort_values(["scenario", "family", "display_name"], na_position="last").reset_index(drop=True)

comparison = attach_seed(build_synthetic_comparison(
    cmdl_root=CMDL_DIR,
    baseline_root=BASELINE_DIR,
    tft_root=TFT_DIR,
    ganet_root=GANET_DIR,
    ablation_root=ABLATION_DIR,
))
recovery_table = build_recovery_table(comparison)
identification_table = build_identification_table(comparison)
compact_summary = aggregate_multiseed(comparison)
significance_tables = build_significance_tables(comparison)

comparison.to_csv(COMPARISON_DIR / "synthetic_comparison.csv", index=False)
recovery_table.to_csv(COMPARISON_DIR / "synthetic_recovery_table.csv", index=False)
identification_table.to_csv(COMPARISON_DIR / "synthetic_identification_table.csv", index=False)
compact_summary.to_csv(COMPARISON_DIR / "synthetic_multiseed_summary.csv", index=False)
for name, frame in significance_tables.items():
    frame.to_csv(COMPARISON_DIR / name, index=False)

tables = {
    "synthetic_comparison": comparison,
    "synthetic_multiseed_summary": compact_summary,
    "synthetic_recovery_table": recovery_table,
    "synthetic_identification_table": identification_table,
    **{Path(name).stem: frame for name, frame in significance_tables.items()},
}
pd.Series({name: len(frame) for name, frame in tables.items()}, name="rows").to_frame()

,rows
synthetic_comparison,280
synthetic_multiseed_summary,14
synthetic_recovery_table,280
synthetic_identification_table,160
synthetic_significance_kstar_mae,12
synthetic_significance_task_loss,12


In [3]:
for name, frame in tables.items():
    print(f"\n=== {name} ({len(frame)} rows) ===")
    display(frame.head(20))


=== synthetic_comparison (280 rows) ===


,family,display_name,experiment,scenario,seed,tracking_backend,best_epoch,best_val_task_loss,task_loss,recon_loss,...,task_mse,task_mae,task_r2,posthoc_kstar_mae,posthoc_kstar_rmse,posthoc_kstar_spearman_rho,posthoc_kstar_spearman_p,posthoc_profile_entropy_mean,posthoc_profile_peak_accuracy,variant
0,ablation,No AC Encoder,no_ac_encoder_linear_seed0,linear,0,json,49,0.084582,0.063652,1.774340e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
1,ablation,No AC Encoder,no_ac_encoder_linear_seed1,linear,1,json,34,0.093879,0.066928,1.199495e+14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
2,ablation,No AC Encoder,no_ac_encoder_linear_seed10,linear,10,json,52,0.092478,0.063951,3.415372e+13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
3,ablation,No AC Encoder,no_ac_encoder_linear_seed11,linear,11,json,55,0.082824,0.064140,3.350851e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
4,ablation,No AC Encoder,no_ac_encoder_linear_seed12,linear,12,json,51,0.086042,0.062684,2.468619e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
5,ablation,No AC Encoder,no_ac_encoder_linear_seed13,linear,13,json,47,0.082942,0.061828,8.545227e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
6,ablation,No AC Encoder,no_ac_encoder_linear_seed14,linear,14,json,75,0.116990,0.052829,8.609338e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
7,ablation,No AC Encoder,no_ac_encoder_linear_seed15,linear,15,json,95,0.120059,0.050631,1.507832e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
8,ablation,No AC Encoder,no_ac_encoder_linear_seed16,linear,16,json,60,0.113241,0.063039,4.523100e+12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder
9,ablation,No AC Encoder,no_ac_encoder_linear_seed17,linear,17,json,56,0.141309,0.068305,1.224071e+11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_ac_encoder



=== synthetic_multiseed_summary (14 rows) ===


,display_name,family,variant,scenario,task_loss_mean,task_loss_std,effective_kstar_mae_mean,effective_kstar_mae_std,effective_kstar_spearman_rho_mean,effective_kstar_spearman_rho_std,effective_lag_entropy_mean_mean,effective_lag_entropy_mean_std,effective_lag_peak_accuracy_mean,effective_lag_peak_accuracy_std,proxy_signal_r2_mean,proxy_signal_r2_std,z_signal_spearman_rho_mean,z_signal_spearman_rho_std,n_seeds,kstar_positive_seed_share
0,No AC Encoder,ablation,no_ac_encoder,linear,0.062946,0.005430,1.931170,0.162463,0.000000,0.000000,2.001045,0.071390,0.13900,0.030245,-1.476138e+16,4.750151e+16,0.000000,0.000000,20,0.0
1,No Recon Regularization,ablation,no_recon_regularization,linear,0.036329,0.004310,1.159246,0.226639,0.944675,0.078489,1.904514,0.085108,0.34700,0.072555,8.993343e-01,9.071034e-02,0.954408,0.081383,20,1.0
2,Uniform Lag,ablation,uniform_lag,linear,0.069512,0.002694,1.912750,0.087577,0.000000,0.000000,2.302585,0.000000,0.00000,0.000000,8.483472e-01,1.571878e-01,0.945730,0.091668,20,0.0
3,CMDL,cmdl,NaN,linear,0.036238,0.004377,1.159093,0.226536,0.944896,0.078103,1.905189,0.085072,0.34675,0.072171,8.995846e-01,9.044959e-02,0.954648,0.080977,20,1.0
4,GA-Net,ganet,NaN,linear,0.032500,0.006861,1.674319,0.087121,0.507661,0.064668,2.243686,0.020126,0.16100,0.026487,NaN,NaN,NaN,NaN,20,1.0
5,Plain LSTM,plain_lstm,NaN,linear,0.069503,0.002897,1.706720,0.090700,0.356054,0.065714,2.148185,0.019232,0.12875,0.018699,NaN,NaN,NaN,NaN,20,1.0
6,TFT,tft,NaN,linear,0.031915,0.002464,1.667532,0.087253,0.471539,0.047113,2.247653,0.008274,0.16650,0.020654,NaN,NaN,NaN,NaN,20,1.0
7,No AC Encoder,ablation,no_ac_encoder,nonlinear,0.066058,0.018424,2.435219,0.260711,0.000000,0.000000,1.820658,0.113915,0.27675,0.150493,-2.850363e+16,5.384774e+16,0.000000,0.000000,20,0.0
8,No Recon Regularization,ablation,no_recon_regularization,nonlinear,0.037843,0.006660,1.466841,0.248370,0.908675,0.204805,1.833409,0.082700,0.48850,0.041330,8.898297e-01,1.934375e-01,0.943500,0.212479,20,1.0
9,Uniform Lag,ablation,uniform_lag,nonlinear,0.093817,0.008114,3.101750,0.097809,0.000000,0.000000,2.302585,0.000000,0.39300,0.027549,8.270833e-01,2.264659e-01,0.911128,0.180868,20,0.0



=== synthetic_recovery_table (280 rows) ===


...

,display_name,family,scenario,experiment,task_loss,proxy_signal_r2,z_signal_spearman_rho,best_epoch
0,Uniform Lag,ablation,linear,uniform_lag_linear_seed11,0.071249,0.953706,0.990887,32
1,Uniform Lag,ablation,linear,uniform_lag_linear_seed12,0.064591,0.953594,0.984676,50
2,CMDL,cmdl,linear,cmdl_linear_seed19,0.039335,0.952217,0.991826,106
3,No Recon Regularization,ablation,linear,no_recon_regularization_linear_seed19,0.039316,0.952216,0.991826,106
4,Uniform Lag,ablation,linear,uniform_lag_linear_seed18,0.070107,0.951380,0.991391,39
5,Uniform Lag,ablation,linear,uniform_lag_linear_seed3,0.068383,0.950381,0.989678,34
6,No Recon Regularization,ablation,linear,no_recon_regularization_linear_seed0,0.031274,0.948780,0.989620,153
7,CMDL,cmdl,linear,cmdl_linear_seed0,0.031284,0.948772,0.989620,153
8,CMDL,cmdl,linear,cmdl_linear_seed10,0.031568,0.947718,0.989809,138
9,No Recon Regularization,ablation,linear,no_recon_regularization_linear_seed10,0.031787,0.947684,0.989809,138



=== synthetic_significance_kstar_mae (12 rows) ===


,scenario,metric,reference,method,n_pairs,reference_mean,method_mean,mean_diff,median_diff,wilcoxon_statistic,wilcoxon_p,greater_is_better,reference_better_mean
0,linear,effective_kstar_mae,CMDL,GA-Net,20,1.159093,1.674319,-0.515226,-0.525680,0.0,0.000002,False,True
1,linear,effective_kstar_mae,CMDL,No AC Encoder,20,1.159093,1.931170,-0.772077,-0.784889,0.0,0.000002,False,True
2,linear,effective_kstar_mae,CMDL,No Recon Regularization,20,1.159093,1.159246,-0.000152,-0.000019,86.0,0.498009,False,True
3,linear,effective_kstar_mae,CMDL,Plain LSTM,20,1.159093,1.706720,-0.547627,-0.587660,0.0,0.000002,False,True
4,linear,effective_kstar_mae,CMDL,TFT,20,1.159093,1.667532,-0.508439,-0.545043,0.0,0.000002,False,True
5,linear,effective_kstar_mae,CMDL,Uniform Lag,20,1.159093,1.912750,-0.753657,-0.798521,0.0,0.000002,False,True
6,nonlinear,effective_kstar_mae,CMDL,GA-Net,20,1.466921,2.689155,-1.222233,-1.226387,0.0,0.000002,False,True
7,nonlinear,effective_kstar_mae,CMDL,No AC Encoder,20,1.466921,2.435219,-0.968298,-0.972136,0.0,0.000002,False,True
8,nonlinear,effective_kstar_mae,CMDL,No Recon Regularization,20,1.466921,1.466841,0.000081,0.000032,80.0,0.368277,False,False
9,nonlinear,effective_kstar_mae,CMDL,Plain LSTM,20,1.466921,2.610799,-1.143878,-1.122322,0.0,0.000002,False,True



=== synthetic_significance_task_loss (12 rows) ===


,scenario,metric,reference,method,n_pairs,reference_mean,method_mean,mean_diff,median_diff,wilcoxon_statistic,wilcoxon_p,greater_is_better,reference_better_mean
0,linear,task_loss,CMDL,GA-Net,20,0.036238,0.032500,0.003737,0.006479,59.0,0.089695,False,False
1,linear,task_loss,CMDL,No AC Encoder,20,0.036238,0.062946,-0.026709,-0.026730,0.0,0.000002,False,True
2,linear,task_loss,CMDL,No Recon Regularization,20,0.036238,0.036329,-0.000091,0.000003,98.0,0.812355,False,True
3,linear,task_loss,CMDL,Plain LSTM,20,0.036238,0.069503,-0.033266,-0.033771,0.0,0.000002,False,True
4,linear,task_loss,CMDL,TFT,20,0.036238,0.031915,0.004322,0.003208,15.0,0.000261,False,False
5,linear,task_loss,CMDL,Uniform Lag,20,0.036238,0.069512,-0.033275,-0.033284,0.0,0.000002,False,True
6,nonlinear,task_loss,CMDL,GA-Net,20,0.037825,0.040765,-0.002940,-0.004405,33.0,0.005581,False,True
7,nonlinear,task_loss,CMDL,No AC Encoder,20,0.037825,0.066058,-0.028233,-0.035216,0.0,0.000002,False,True
8,nonlinear,task_loss,CMDL,No Recon Regularization,20,0.037825,0.037843,-0.000018,0.000002,85.0,0.474905,False,True
9,nonlinear,task_loss,CMDL,Plain LSTM,20,0.037825,0.092684,-0.054859,-0.054313,0.0,0.000002,False,True
